# Week 13 — Frozen Model and Inference Pipeline

## Goal

Turn the selected Configuration D EEGNet from an experimental notebook model
into a reproducible model that can be saved, loaded, and used to make predictions
on new preprocessed EEG trials.

This week does not involve further model selection or hyperparameter tuning.

The primary model is frozen as Configuration D:

- EEGNet architecture
- learning rate: 1e-3
- dropout: 0.50
- weight decay: 1e-4
- batch size: 32
- maximum 30 epochs
- no max-norm constraints

Seed 42 will be used as the reproducible deployment checkpoint.

The held-out test subjects remain untouched.

### Imports and paths

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score


ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"

MODELS_DIR.mkdir(exist_ok=True)


X = np.load(DATA_DIR / "X_motor_imagery.npy")
y = np.load(DATA_DIR / "y_motor_imagery.npy")
subjects = np.load(DATA_DIR / "subjects.npy")

split_df = pd.read_csv(
    RESULTS_DIR / "week9_subject_split.csv"
)

train_subjects = split_df.loc[
    split_df["split"] == "train",
    "subject"
].to_numpy()

val_subjects = split_df.loc[
    split_df["split"] == "validation",
    "subject"
].to_numpy()

train_indices = np.where(
    np.isin(subjects, train_subjects)
)[0]

val_indices = np.where(
    np.isin(subjects, val_subjects)
)[0]


print("X:", X.shape)
print("Train trials:", len(train_indices))
print("Validation trials:", len(val_indices))
print("Labels:", np.unique(y))

X: (4898, 64, 481)
Train trials: 3423
Validation trials: 718
Labels: [0 1]


### Datasets

In [2]:
class EEGDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        trial_index = self.indices[index]

        x = torch.tensor(
            self.X[trial_index],
            dtype=torch.float32
        )

        x = x * 1e6
        x = x.unsqueeze(0)

        target = torch.tensor(
            self.y[trial_index],
            dtype=torch.long
        )

        return x, target

In [3]:
train_dataset = EEGDataset(
    X,
    y,
    train_indices
)

val_dataset = EEGDataset(
    X,
    y,
    val_indices
)

### EEGNet Config D

In [4]:
class EEGNet(nn.Module):
    def __init__(
        self,
        n_channels=64,
        n_classes=2,
        F1=8,
        D=2,
        F2=16,
        temporal_kernel=40,
        separable_kernel=16,
        dropout_p=0.5
    ):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(
                1,
                F1,
                (1, temporal_kernel),
                padding="same",
                bias=False
            ),
            nn.BatchNorm2d(F1),

            nn.Conv2d(
                F1,
                F1 * D,
                (n_channels, 1),
                groups=F1,
                bias=False
            ),
            nn.BatchNorm2d(F1 * D),

            nn.ELU(),

            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_p)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
                F1 * D,
                F1 * D,
                (1, separable_kernel),
                padding="same",
                groups=F1 * D,
                bias=False
            ),

            nn.Conv2d(
                F1 * D,
                F2,
                (1, 1),
                bias=False
            ),

            nn.BatchNorm2d(F2),
            nn.ELU(),

            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_p)
        )

        self.classifier = nn.Linear(
            F2 * 15,
            n_classes
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)

        x = torch.flatten(
            x,
            start_dim=1
        )

        return self.classifier(x)

### Check

In [5]:
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(DEVICE)

mps


### Train epoch function

In [6]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):
    model.train()

    all_targets = []
    all_predictions = []

    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)

        predictions = logits.argmax(dim=1)

        all_targets.extend(
            y_batch.cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

    loss = total_loss / len(loader.dataset)

    ba = balanced_accuracy_score(
        all_targets,
        all_predictions
    )

    return loss, ba

### Evaluation function

In [7]:
def evaluate(
    model,
    loader,
    criterion
):
    model.eval()

    all_targets = []
    all_predictions = []

    total_loss = 0.0

    with torch.inference_mode():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = model(X_batch)

            loss = criterion(
                logits,
                y_batch
            )

            total_loss += loss.item() * len(y_batch)

            predictions = logits.argmax(dim=1)

            all_targets.extend(
                y_batch.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    loss = total_loss / len(loader.dataset)

    ba = balanced_accuracy_score(
        all_targets,
        all_predictions
    )

    return loss, ba

### Predetermined training run

In [8]:
SEED = 42
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 30


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


generator = torch.Generator()
generator.manual_seed(SEED)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


model = EEGNet(
    dropout_p=0.5
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


best_val_ba = -np.inf
best_epoch = None
best_state = {}


for epoch in range(1, EPOCHS + 1):

    train_loss, train_ba = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_ba = evaluate(
        model,
        val_loader,
        criterion
    )

    if val_ba > best_val_ba:
        best_val_ba = val_ba
        best_epoch = epoch

        best_state = {}

        for key, value in model.state_dict().items():
            best_state[key] = (
                value
                .detach()
                .cpu()
                .clone()
            )

    print(
        f"Epoch {epoch:02d} | "
        f"train BA {train_ba:.3f} | "
        f"val BA {val_ba:.3f}"
    )

/Users/heihei/Desktop/neurosignallab/.venv/lib/python3.13/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1105.)
  return F.conv2d(


Epoch 01 | train BA 0.500 | val BA 0.511
Epoch 02 | train BA 0.500 | val BA 0.511
Epoch 03 | train BA 0.510 | val BA 0.507
Epoch 04 | train BA 0.495 | val BA 0.527
Epoch 05 | train BA 0.530 | val BA 0.540
Epoch 06 | train BA 0.549 | val BA 0.575
Epoch 07 | train BA 0.558 | val BA 0.589
Epoch 08 | train BA 0.581 | val BA 0.550
Epoch 09 | train BA 0.577 | val BA 0.586
Epoch 10 | train BA 0.596 | val BA 0.593
Epoch 11 | train BA 0.598 | val BA 0.606
Epoch 12 | train BA 0.608 | val BA 0.602
Epoch 13 | train BA 0.610 | val BA 0.600
Epoch 14 | train BA 0.600 | val BA 0.623
Epoch 15 | train BA 0.611 | val BA 0.593
Epoch 16 | train BA 0.602 | val BA 0.603
Epoch 17 | train BA 0.612 | val BA 0.603
Epoch 18 | train BA 0.621 | val BA 0.618
Epoch 19 | train BA 0.625 | val BA 0.628
Epoch 20 | train BA 0.619 | val BA 0.615
Epoch 21 | train BA 0.618 | val BA 0.613
Epoch 22 | train BA 0.628 | val BA 0.622
Epoch 23 | train BA 0.636 | val BA 0.619
Epoch 24 | train BA 0.631 | val BA 0.605
Epoch 25 | train

In [9]:
print("Best epoch:", best_epoch)
print("Best validation BA:", best_val_ba)

Best epoch: 19
Best validation BA: 0.6277011922503726


### Checkpoint 

In [10]:
model.load_state_dict(best_state)
model.eval()

EEGNet(
  (block1): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 40), stride=(1, 1), padding=same, bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): Conv2d(8, 16, kernel_size=(64, 1), stride=(1, 1), groups=8, bias=False)
    (3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (4): ELU(alpha=1.0)
    (5): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (6): Dropout(p=0.5, inplace=False)
  )
  (block2): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=same, groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.5, inplace=False)
  )
  (classifier): Linear(in_features=240, out_feat

In [11]:
checkpoint = {
    "model_state_dict": best_state,

    "model_name": "EEGNet_Config_D",

    "seed": SEED,
    "best_epoch": best_epoch,
    "validation_balanced_accuracy": best_val_ba,

    "n_channels": 64,
    "n_classes": 2,
    "n_times": 481,
    "sampling_rate_hz": 160,

    "input_scale": "volts_to_microvolts",
    "input_multiplier": 1e6,

    "preprocessing": {
        "bandpass_hz": [7, 30],
        "reference": "common_average",
        "epoch_window_seconds": [1, 4]
    }
}


checkpoint_path = (
    MODELS_DIR /
    "config_d_seed42_best.pt"
)

torch.save(
    checkpoint,
    checkpoint_path
)

print(checkpoint_path)

/Users/heihei/Desktop/neurosignallab/models/config_d_seed42_best.pt


### Add checkpoint metadata after inspecting old notebook

In [12]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

checkpoint["class_names"] = {
    0: "imagined left fist",
    1: "imagined right fist"
}

torch.save(
    checkpoint,
    checkpoint_path
)

print(checkpoint["class_names"])

{0: 'imagined left fist', 1: 'imagined right fist'}


### First inferecnce

In [13]:
trial_index = val_indices[0]

trial = X[trial_index]

print(trial.shape)

(64, 481)


In [14]:
x = torch.tensor(
    trial,
    dtype=torch.float32
)

x = x * 1e6

x = x.unsqueeze(0)
x = x.unsqueeze(0)

x = x.to(DEVICE)

print(x.shape)

torch.Size([1, 1, 64, 481])


In [15]:
model.eval()

with torch.inference_mode():

    logits = model(x)

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    prediction = probabilities.argmax(
        dim=1
    )

### Inspect inference

In [16]:
print(
    "Logits:",
    logits.cpu().numpy()
)

print(
    "Probabilities:",
    probabilities.cpu().numpy()
)

print(
    "Prediction:",
    prediction.item()
)

print(
    "True label:",
    y[trial_index]
)

Logits: [[ 0.06599953 -0.08815247]]
Probabilities: [[0.53846186 0.46153814]]
Prediction: 0
True label: 1


### Check inference src

In [17]:
import sys

src_path = str(ROOT / "src")

if src_path not in sys.path:
    sys.path.append(src_path)


from neurosignallab.inference import (
    load_eegnet,
    predict_trial
)

In [18]:
checkpoint_path = (
    MODELS_DIR /
    "config_d_seed42_best.pt"
)


inference_model, checkpoint, inference_device = (
    load_eegnet(checkpoint_path)
)


print("Device:", inference_device)
print("Model:", checkpoint["model_name"])
print("Best epoch:", checkpoint["best_epoch"])
print(
    "Validation BA:",
    checkpoint["validation_balanced_accuracy"]
)

Device: mps
Model: EEGNet_Config_D
Best epoch: 19
Validation BA: 0.6277011922503726


In [19]:
trial_index = val_indices[0]

trial = X[trial_index]

result = predict_trial(
    inference_model,
    trial,
    checkpoint,
    inference_device
)


print("Logits:", result["logits"])
print(
    "Probabilities:",
    result["probabilities"]
)
print("Prediction:", result["prediction"])
print("Predicted class:", result["predicted_class"])
print("True label:", y[trial_index])


Logits: [ 0.06599953 -0.08815247]
Probabilities: [0.53846186 0.46153814]
Prediction: 0
Predicted class: imagined left fist
True label: 1


## Week 13 Conclusion

The selected Configuration D EEGNet was converted from an experimental notebook
model into a reproducible inference pipeline.

The seed-42 model was retrained using the frozen Configuration D specification
and exactly reproduced the previous validation result:

- best epoch: `19`
- validation balanced accuracy: `0.627701`

The best model state was saved as a reusable checkpoint containing both learned
weights and relevant model/preprocessing metadata.

The EEGNet architecture was moved into `src/neurosignallab/models.py`, while
model loading and single-trial prediction were implemented in
`src/neurosignallab/inference.py`.

The reusable inference pipeline successfully reproduced the exact logits,
probabilities, and prediction produced by the notebook implementation.

The class mapping was also verified directly from the preprocessing pipeline:

- `0` = imagined left fist
- `1` = imagined right fist

This means NeuroSignalLab now has a frozen model that can be loaded and used for
predictions without retraining or depending on notebook state.

The held-out test set remains untouched.